## Tugas

Prediksi apakah atlet dengan data berikut masuk draft yes/no:
1. input = [7.5, 7.0]
2. input = [4.0, 6.5]

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv('athletes.csv')

X = df[['speed', 'agility']].values
y = df['draft'].values

en = LabelEncoder()
y = en.fit_transform(y)

In [3]:
# Model KNN dengan k = 3 (akurasi terbaik), dilatih memakai seluruh data atlet
knn_atlet = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn_atlet.fit(X, y)

atlet_baru = np.array([[7.5, 7.0],
                       [4.0, 6.5]])

pred = knn_atlet.predict(atlet_baru)
hasil = en.inverse_transform(pred)

# Lihat 3 tetangga terdekat dari masing-masing input
jarak, idx = knn_atlet.kneighbors(atlet_baru)

for i, data in enumerate(atlet_baru):
    print(f"Input {data} -> draft: {hasil[i]}")
    print("   Tetangga terdekat (id atlet):", df['id'].values[idx[i]],
          "| label:", en.inverse_transform(y[idx[i]]))

Input [7.5 7. ] -> draft: yes
   Tetangga terdekat (id atlet): [19 20 13] | label: ['yes' 'yes' 'no']
Input [4.  6.5] -> draft: no
   Tetangga terdekat (id atlet): [15  7  2] | label: ['yes' 'no' 'no']


#### Penjelasan:
Kode di atas untuk memprediksi dua atlet baru. Saya pakai **k = 3** karena di percobaan sebelumnya akurasinya paling tinggi, dan modelnya dilatih ulang dengan **seluruh 20 data** supaya punya lebih banyak tetangga sebagai acuan. Hasil angkanya (0/1) diubah lagi ke teks pakai `en.inverse_transform`, dan `kneighbors()` dipakai untuk melihat siapa saja tetangga terdekatnya, jadi alasan di balik prediksinya kelihatan.

Hasilnya:
1. Input `[7.5, 7.0]` -> **yes (masuk draft)**. Tetangga terdekatnya atlet 19 (yes), 20 (yes), dan 13 (no), jadi 2 dari 3 tetangga bilang `yes`. Atlet 13 itu titik merah yang nyasar tadi, tetapi tidak cukup untuk mengalahkan dua tetangga lainnya.
2. Input `[4.0, 6.5]` -> **no (tidak masuk draft)**. Tetangga terdekatnya atlet 15 (yes), 7 (no), dan 2 (no), jadi 2 dari 3 tetangga bilang `no`.

Untuk dataset iris, prediksi input data berikut ini masuk ke species apa:
1. input_data = [[7.1, 1.2, 5.0, 2.0]]
2. input_data = [[5.1, 3.5, 1.4, 0.2]]
3. input_data = [[6.4, 3.7, 4.2, 2.2]]

In [5]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier

# Load dataset Iris
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target   # sudah berupa angka 0/1/2

knn_iris = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn_iris.fit(X_iris, y_iris)

input_data = [[7.1, 1.2, 5.0, 2.0],
              [5.1, 3.5, 1.4, 0.2],
              [6.4, 3.7, 4.2, 2.2]]

input_df = pd.DataFrame(input_data, columns=iris.feature_names)
pred_iris = knn_iris.predict(input_df)

# Lihat kelas dari 3 tetangga terdekat masing-masing input
jarak, idx = knn_iris.kneighbors(input_df)

for i, data in enumerate(input_data):
    print(f"Input {data} -> spesies: {iris.target_names[pred_iris[i]]}")
    print("   Kelas 3 tetangga terdekat:", iris.target_names[y_iris[idx[i]]])

Input [7.1, 1.2, 5.0, 2.0] -> spesies: virginica
   Kelas 3 tetangga terdekat: ['versicolor' 'virginica' 'virginica']
Input [5.1, 3.5, 1.4, 0.2] -> spesies: setosa
   Kelas 3 tetangga terdekat: ['setosa' 'setosa' 'setosa']
Input [6.4, 3.7, 4.2, 2.2] -> spesies: versicolor
   Kelas 3 tetangga terdekat: ['versicolor' 'versicolor' 'versicolor']


#### Penjelasan:
Data Iris saya muat ulang dulu supaya tidak tertimpa variabel `X` dan `y` dari dataset athletes. Model KNN (k = 3) dilatih pakai seluruh 150 data, lalu tiga input baru diprediksi. Input dibuat sebagai DataFrame dengan nama kolom yang sama seperti data latih supaya tidak muncul peringatan dari Scikit-Learn.

Hasilnya:
1. `[7.1, 1.2, 5.0, 2.0]` -> **virginica**. Dua dari tiga tetangganya virginica, satu versicolor. Ini bukan prediksi yang bulat 3-0, karena data ini punya `sepal width` 1.2 yang nilainya jauh lebih kecil dari data Iris pada umumnya, tetapi `petal length` 5.0 dan `petal width` 2.0-nya lebih dekat ke ciri virginica.
2. `[5.1, 3.5, 1.4, 0.2]` -> **setosa**. Ketiga tetangganya setosa. Malah datanya sama persis dengan data pertama di dataset (jarak 0), jadi wajar kalau hasilnya yakin.
3. `[6.4, 3.7, 4.2, 2.2]` -> **versicolor**. Ketiga tetangganya versicolor, jadi prediksinya cukup yakin.

**Kesimpulan yang saya dapatkan**

KNN menebak kelas dari **K tetangga terdekat** memakai jarak Euclidean. Pada dataset Iris, model dengan k = 3 mencapai akurasi sekitar 96,67% karena tiga spesies itu cukup terpisah, terutama setosa. Pada dataset athletes, nilai K berpengaruh besar terhadap hasil: K = 3 dan 4 memberi akurasi terbaik (1.0), sedangkan K yang terlalu besar (K = 10) justru menurunkan akurasi jadi 0.5. Jadi nilai K perlu dicoba dan dibandingkan dulu, tidak bisa asal ditentukan.